# Chapter 18 &mdash; Eager versus Lazy Evaluation, and the Combinator $Y_e$

**Concept 8 of the Chapter 18 decomposition:** *Eager versus Lazy Evaluation, and the Combinator $Y_e$*

Python evaluates arguments first, so $Y$ diverges; $Y_e$ wraps the self-application in a lambda.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Eager-Versus-Lazy/Concept-Eager-Versus-Lazy.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The difference between $Y$ working and $Y$ hanging is **evaluation order**.

* **Eager** (call-by-value: Python, ML, Scheme) &mdash; evaluate the argument, *then*
  apply. So in $f\,(x\,x)$, the $x\,x$ is evaluated first, and it expands to $x\,x$
  again. Infinite regress.
* **Lazy** (call-by-name/need: Haskell) &mdash; pass the argument unevaluated. $x\,x$ is
  only forced if $f$ actually uses it, and the base case does not.

The fix is **eta-expansion**: replace $x\,x$ by $\lambda v.\,(x\,x)\,v$. By $\eta$
these are the same *function*, but the second is a **value** &mdash; a lambda &mdash; so an
eager language stops there.

$$Y_e = \lambda f.\,(\lambda x.\,f\,(\lambda v.\,x\,x\,v))\,(\lambda x.\,f\,(\lambda v.\,x\,x\,v))$$

Same combinator, one wrapper. This is why every strict language's "Y combinator" has
that extra lambda.

## 2. Definitions

### Y, Y_e, and a way to see the evaluation order

In [ ]:
# --- fixpoint combinators ------------------------------------------------
# Y diverges under Python's EAGER evaluation, because (x x) is evaluated
# before it is needed.  Y_e ("eager Y", also called Z) wraps the
# self-application in a lambda, delaying it until it is applied.
Y  = lambda f: (lambda x: f(x(x)))(lambda x: f(x(x)))          # loops in Python
Ye = lambda f: (lambda x: f(lambda v: x(x)(v)))(lambda x: f(lambda v: x(x)(v)))


TRACE = []
def traced(name, v):
    TRACE.append(name); return v

### Eager and lazy, simulated

In [ ]:
def eager_apply(f, arg_thunk):
    v = arg_thunk()                 # evaluate FIRST
    return f(lambda: v)

def lazy_apply(f, arg_thunk):
    return f(arg_thunk)             # pass the thunk, unevaluated

## 3. Tests

Eager evaluation forces the argument even when it is unused.

In [ ]:
TRACE.clear()
def ignore(thunk): return 'done'
eager_apply(ignore, lambda: traced('evaluated', 42))
print("eager : trace =", TRACE)
TRACE.clear()
lazy_apply(ignore, lambda: traced('evaluated', 42))
print("lazy  : trace =", TRACE)
assert TRACE == []
print("\nThe lazy version never evaluated the argument, because ignore")
print("never used it.  That is exactly what Y needs.")

So **$Y$ diverges** under Python's eager rule.

In [ ]:
import sys
G = lambda f: lambda n: 1 if n == 0 else n * f(n - 1)
sys.setrecursionlimit(200)
try:
    Y(G); print("returned (unexpected)")
except RecursionError:
    print("Y(G) : RecursionError -- (x x) expands forever before f is applied")
sys.setrecursionlimit(3000)

**$Y_e$** eta-expands the self-application, turning it into a value.

In [ ]:
fact = Ye(G)
print("Ye(G) :", [fact(n) for n in range(8)])
assert fact(6) == 720
print()
print("  Y  : ... f (x x) ...          (x x) is evaluated eagerly -> loop")
print("  Ye : ... f (lambda v: x x v)  a LAMBDA -- already a value -> stop")

By $\eta$ the two are the same function; only the **timing** differs.

In [ ]:
h = lambda n: n * 2
h_eta = lambda v: h(v)
print("  h and its eta-expansion agree :", all(h(n) == h_eta(n) for n in range(5)))
assert all(h(n) == h_eta(n) for n in range(5))
print("\nSame function, different evaluation behaviour.  Eta is sound for")
print("VALUES and changes WHEN work happens -- which is the whole fix.")

A lazy language needs no fix &mdash; simulated with thunks.

In [ ]:
def lazy_Y(G, n, depth=0):
    # call-by-name: build the recursive call as a thunk
    def rec(k): return lazy_Y(G, k, depth + 1)
    return G(rec)(n)
print("lazy-style Y :", [lazy_Y(G, n) for n in range(8)])
assert lazy_Y(G, 6) == 720
print("\nHaskell's `fix` is literally `fix f = f (fix f)` -- no eta needed.")

The rule to remember.

In [ ]:
print("strict language  -> use Ye (a.k.a. the Z combinator)")
print("lazy language    -> plain Y works")
print()
print("and if your Y hangs, look for a missing eta-expansion.")

## 4. Exercises


1. Write $Y_e$ for a two-argument recursive function. Where does the extra $v$ go?
2. Does Scheme's `letrec` use $Y$? What does it use instead?
3. Show that eta-expansion is unsound in a language with side effects.

In [ ]:
# Your work for the exercises above.